# Usecase 3: Absolute abundance prediction — run ritme models

The notebook launches ritme experiments via the shared template `src/run_ritme_model.sh` and the `submit_model` helper in `src/launch_models.py`. It is written against ritme >= 1.3.0 (tested with v1.4.0). Set up the env once (the last command must be run from the repo root):

```shell
mamba create -n ritme_usecases -c adamova -c conda-forge -c bioconda -c pytorch ritme ipykernel nbconvert -y
conda activate ritme_usecases
pip install -e .
```

Two execution modes:
- `mode="slurm"` submits each model class as its own sbatch job using the SLURM resource defaults in `src/launch_models.py:MODEL_RESOURCES`.
- `mode="local"` runs the template inline in the current process — useful for short smoke tests.

The template performs (idempotently): QIIME2 `.qza` → `.tsv`/`.nwk` conversion → train/test split → `find-best-model-config` → `evaluate-tuned-models`.

## Setup

In [ ]:
from src.launch_models import submit_model

## Configuration

In [ ]:
# Where ritme writes experiment outputs. The launcher creates this dir
# (relative to the repo root if not absolute) and stores per-experiment
# subfolders + sbatch logs under it. Override for cluster runs, e.g.
# ``"/cluster/work/<lab>/<user>/ritme_runs_v140"``.
LOGS_DIR = "use_cases/ritme_runs/local"

# Cluster account (SLURM ``--account=...``, a.k.a. "share") forwarded
# to every sbatch submission below. ``None`` (default) leaves the flag
# unset and uses the cluster default; set to your own, e.g. ``"my_lab"``.
SLURM_ACCOUNT = None

# Per-model ritme search budget (seconds) — caps `find-best-model-config`.
# SHAP + bootstrap run afterwards (minutes on small data, low single-digit
# hours on the largest cases). In slurm mode, `submit_model` snaps the
# walltime to the smallest cluster tier (4 h / 24 h / 120 h) that fits
# this budget plus a 1 h SHAP/bootstrap buffer; see
# `src/launch_models.py:SLURM_WALLTIME_TIERS_S`. Pass `slurm_time=`
# directly on each submit_model call to override.
TIME_BUDGET_S = 406800

# Per-job SLURM resources for one sbatch job per model class (slurm mode
# only; ignored when mode="local"). Leave as None to inherit
# `src/launch_models.py:SLURM_RESOURCES[(usecase, model)]`.
CPUS = None
MEM_PER_CPU_MB = None

# Ritme Ray-Tune concurrency (parallel trials per model class). Applies
# in both modes. Leave as None to inherit
# `src/launch_models.py:MAX_CONCURRENT_TRIALS[model]`.
MAX_CONCURRENT_TRIALS = None

## Launch experiments

Each call below submits one model class. Replace `mode="local"` with `mode="slurm"` to dispatch as separate cluster jobs (using the per-model resources in `MODEL_RESOURCES`). Override `time_budget_s` etc. by editing the matching JSON in `config/`.

In [ ]:
models = ["linreg", "rf", "xgb", "nn_reg"]

common = dict(
    mode="slurm",
    logs_dir=LOGS_DIR,
    slurm_account=SLURM_ACCOUNT,
    cpus=CPUS,
    mem_per_cpu_mb=MEM_PER_CPU_MB,
    max_concurrent_trials=MAX_CONCURRENT_TRIALS,
    config_overrides={"time_budget_s": TIME_BUDGET_S},
)
for model_type in models:
    submit_model("u3_legacy", model_type, **common)

## Launch experiments — without metadata enrichment

Same model classes as above, but launched with `variant="no_enrich"`. This synthetic variant derives from the per-usecase base config and strips `model_hyperparameters.data_enrich_with` so ritme runs without metadata enrichment. Outputs land under `<prefix>_<model>_<sampler>_no_enrich`, separately from the enriched runs above, so the two sets can be compared directly via `evaluate_all_trials.ipynb`.

In [ ]:
models = ["linreg", "rf", "xgb", "nn_reg"]

for model_type in models:
    submit_model("u3_legacy", model_type, variant="no_enrich", **common)

## Optional variants

In [ ]:
# `xgb` was also explored with seeded warm-start configurations:
# submit_model("u3_legacy", "xgb", variant="w_start", mode="slurm", logs_dir=LOGS_DIR, slurm_account=SLURM_ACCOUNT)